In [1]:
import numpy as np
import librosa
import os

In [2]:
sample_rate = 16000
mel_bands = 32

def extract_rf_features(audio_data, sr):
    """
    Extracts a 1D vector of statistical features for the Random Forest.
    This captures the 'mathematical' structure of the sound.
    """
    mfcc = np.mean(librosa.feature.mfcc(y=audio_data, sr=sr, n_mfcc=40).T, axis=0)
    centroid = np.mean(librosa.feature.spectral_centroid(y=audio_data, sr=sr))
    rolloff = np.mean(librosa.feature.spectral_rolloff(y=audio_data, sr=sr))
    bandwidth = np.mean(librosa.feature.spectral_bandwidth(y=audio_data, sr=sr))
    contrast = np.mean(librosa.feature.spectral_contrast(y=audio_data, sr=sr).T, axis=0)
    zcr = np.mean(librosa.feature.zero_crossing_rate(audio_data))
    rms = np.mean(librosa.feature.rms(y=audio_data))
    chroma = np.mean(librosa.feature.chroma_stft(y=audio_data, sr=sr).T, axis=0)
    
    return np.hstack([mfcc, centroid, rolloff, bandwidth, contrast, zcr, rms, chroma])

def extract_mel_features(audio_file):
    """
    Returns TWO lists: 
    1. Mel Spectrograms (Images for CNN)
    2. RF Features (Stats for Random Forest)
    """
    y, sr = librosa.load(audio_file, sr=sample_rate)
    mel_features = []
    rf_features = []

    window_samples = 5 * sample_rate         # 5 second window
    hop_samples = int(1.0 * sample_rate)     # 1 second hop for variety
    target_width = 157 

    def process_and_add(audio_data):
        # 1. Ensure correct length
        if len(audio_data) < window_samples:
            audio_data = np.pad(audio_data, (0, window_samples - len(audio_data)), mode='constant')
        else:
            audio_data = audio_data[:window_samples]
            
        # 2. Generate Mel Spectrogram
        mel = librosa.feature.melspectrogram(y=audio_data, sr=sample_rate, n_mels=mel_bands)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        
        # 3. Standardize width to 157
        if mel_db.shape[1] > target_width:
            mel_db = mel_db[:, :target_width]
        elif mel_db.shape[1] < target_width:
            mel_db = np.pad(mel_db, ((0,0), (0, target_width - mel_db.shape[1])))
            
        # 4. Per-Segment Normalization 
        mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-8)
        mel_features.append(mel_db)

        # 5. Extract and add RF features
        rf_vec = extract_rf_features(audio_data, sample_rate)
        rf_features.append(rf_vec)
        
    # Main Loop: Slides through the files
    for start in range(0, len(y), hop_samples):
        seg = y[start:start + window_samples]
        
        # Skip segments that are too short/mostly silence
        if len(seg) < (0.5 * sample_rate):
            continue

        process_and_add(seg)
        process_and_add(seg + np.random.normal(0, 0.005, len(seg))) # Noise
        process_and_add(seg * np.random.uniform(0.6, 1.4))          # Gain
        process_and_add(np.roll(seg, int(len(seg) * 0.4)))          # Time Shift
        process_and_add(librosa.effects.pitch_shift(seg, sr=sample_rate, n_steps=1.5)) # Pitch
        
    return mel_features, rf_features

In [3]:
# EXTRACTION AND SAVING LOOP
os.makedirs("features", exist_ok=True)

classes = ['Noises/appliance', 'Noises/fire', 'Noises/siren', 'Noises/carbon']

for class_name in classes:
    directory = f'./{class_name}'
    
    if not os.path.exists(directory):
        print(f"⚠️ Skipping {directory} (Folder not found)")
        continue
        
    for audio_file in os.listdir(directory):
        if audio_file.lower().endswith(('.wav')):
            print(f"Extracting {audio_file}...")
            
            # Unpack both the images and the math stats
            mels, rfs = extract_mel_features(f"{directory}/{audio_file}")
            
            # Remove the file extension .wav (gets 'fire_001' from 'fire_001.wav')
            base_name = audio_file.rsplit('.', 1)[0]

            # Save everything
            for i in range(len(mels)):
                np.save(f'features/{base_name}_{i}.npy', mels[i])
                np.save(f'features/{base_name}_{i}_rf.npy', rfs[i])

print("\n✅ All features extracted and saved successfully!")

Extracting appliance_m6Q53Ouf0AM_30.0-40.0.wav...


/opt/anaconda3/envs/Random/lib/python3.11/site-packages/librosa/core/intervals.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


Extracting appliance_tS4R6BdMYxs_0.0-10.0.wav...
Extracting appliance_NvT6ypvmQQk_0.0-10.0.wav...
Extracting appliance_J8kp0xDt6IQ_0.0-9.0.wav...
Extracting appliance_ICLkuwWO9tU_30.0-40.0.wav...
Extracting appliance_yYdYMd4RX_w_30.0-40.0.wav...
Extracting appliance_z8Dy5CNoNxc_0.0-10.0.wav...
Extracting appliance_1lBl4WXrFxE_30.0-40.0.wav...
Extracting appliance_rF-N3jUEwHw_0.0-10.0.wav...
Extracting appliance_ismVkDVx3cs_20.0-30.0.wav...
Extracting appliance_XprqiWeGihg_7.0-17.0.wav...
Extracting appliance_ZbtRzrZxr7A_19.0-29.0.wav...
Extracting appliance_XuS3-gORs-o_30.0-40.0.wav...
Extracting appliance_kjC2NZ833BQ_16.0-26.0.wav...
Extracting appliance_zlVvYUGTu94_6.0-16.0.wav...
Extracting appliance_GSaEuUDUmk8_25.0-35.0.wav...
Extracting appliance_c9_MK8mMnO8_6.0-16.0.wav...
Extracting appliance_WnICFGLRdYg_0.0-10.0.wav...
Extracting appliance_89DYioGt-QU_29.0-39.0.wav...
Extracting appliance_FzT-vZ7e94Q_10.0-20.0.wav...
Extracting appliance_PIdcZV3BNHU_10.0-20.0.wav...
Extracting

/opt/anaconda3/envs/Random/lib/python3.11/site-packages/librosa/core/pitch.py:101: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(


Extracting appliance_hWE5m0uXS4A_6.0-16.0.wav...
Extracting appliance_DhvfX3kALh0_7.0-17.0.wav...
Extracting appliance_1hJB8Fg5UKI_3.0-13.0.wav...
Extracting appliance_ptIHZv3KdJw_0.0-3.0.wav...
Extracting appliance_0DefiYiEF4E_19.0-29.0.wav...
Extracting appliance_Qg7U2fFyFVI_0.0-10.0.wav...
Extracting appliance_85OY8T4UCkw_6.0-16.0.wav...
Extracting appliance_NK92DUyyngc_30.0-40.0.wav...
Extracting appliance_U7GdoSpUb7Y_50.0-60.0.wav...
Extracting appliance_yeXO1LOKPMg_0.0-10.0.wav...
Extracting appliance_Zob-z2eLj_4_10.0-20.0.wav...
Extracting appliance_Zgc7p3BZKxc_0.0-10.0.wav...
Extracting appliance_RoYij4guSkE_30.0-40.0.wav...
Extracting appliance_dJMmAHoLNY4_20.0-30.0.wav...
Extracting appliance_MuX_WLoOItE_18.0-28.0.wav...
Extracting appliance_4SZjz5UzwrI_30.0-40.0.wav...
Extracting appliance_3q5eb-iSmJg_30.0-40.0.wav...
Extracting appliance_poz8FFcAmNY_0.0-10.0.wav...
Extracting appliance_NKbnGasaVlc_11.0-21.0.wav...
Extracting appliance_6gELF9ByMCc_5.0-15.0.wav...
Extracting 